# Global Data Science Salaries and Trends

## Description:
This dataset provides a comprehensive overview of global data science salaries and employment trends from 2020 to 2022 (quantity:607). It includes details on job titles, experience levels, remote work ratios, and salary distributions across various currencies and regions. Perfect for analysis and insights into the data science job market (acc:64%).

1. salary_in_usd: Salary converted to USD, unit:k$
2. https://www.kaggle.com/datasets/zain280/data-science-salaries

### Import necessary packages

In [5]:
import os
import numpy as np
import pandas as pd
# 💡 完全对齐规范：直接从外部工具箱导入标准的 12 参数 plot_cpp，本地绝不重复定义
from FL_cpp_method import analyze_dataset, plot_cpp

In [6]:
# %%
def dirichlet_continuous_allocation(Y, Yhat, alpha_dir, num_clients, min_samples_per_client=15, num_bins=5):
    """
    针对连续型回归数据的 alpha_Dir 非独立同分布联邦切分算法。
    """
    pseudo_classes = pd.qcut(Y, q=num_bins, labels=False, duplicates='drop')
    classes = np.unique(pseudo_classes)
    num_classes = len(classes)
    
    class_indices = {c: np.where(pseudo_classes == c)[0] for c in classes}
    for c in classes:
        np.random.shuffle(class_indices[c])
        
    client_indices = [[] for _ in range(num_clients)]
    
    # 阶段一：分配基础最低保有量保底
    base_per_class = min_samples_per_client // num_classes
    if base_per_class < 1: base_per_class = 1
        
    for c in classes:
        indices = class_indices[c]
        available = len(indices)
        required = base_per_class * num_clients
        actual_base = base_per_class if required <= available else available // num_clients
        
        if actual_base > 0:
            for i in range(num_clients):
                client_indices[i].extend(indices[i * actual_base : (i + 1) * actual_base])
            class_indices[c] = indices[num_clients * actual_base:]
            
    # 阶段二：由分配矩阵驱动的自由异质分发
    for c in classes:
        indices = class_indices[c]
        if len(indices) == 0: continue
            
        proportions = np.random.dirichlet([alpha_dir] * num_clients)
        counts = np.floor(proportions * len(indices)).astype(int)
        
        remainder = len(indices) - np.sum(counts)
        for _ in range(remainder):
            counts[np.random.choice(num_clients)] += 1
            
        start = 0
        for i in range(num_clients):
            end = start + counts[i]
            # 🛠️ 语法修正：移除了之前笔误残留的重复单值 extend
            client_indices[i].extend(indices[start:end])
            start = end
            
    reordered_indices = []
    actual_sizes = []
    for i in range(num_clients):
        np.random.shuffle(client_indices[i])
        reordered_indices.extend(client_indices[i])
        actual_sizes.append(len(client_indices[i]))
        
    return Y[reordered_indices], Yhat[reordered_indices], actual_sizes

In [7]:
# %%
# ==============================================================================
# 1. 实验控制元参数初始化（薪资分位数推断任务：alpha=0.05 对应 95% 置信区间）
# ==============================================================================
dataset_name = 'salary'
alpha = 0.05  
method = "quantile"  
num_clients = 20  
xlim = [30, 200]  
ylim = [0, 1.0]

iid_total_records = []
non_iid_total_records = []
acc_steps = np.arange(0.1, 1.1, 0.1)

# ==============================================================================
# 2. 纵向多精度大循环核心
# ==============================================================================
for acc in acc_steps:
    acc_str = f"{acc:.1f}"
    data_path = f'../data/{dataset_name}/{dataset_name}_acc_{acc_str}.npz'
    
    if not os.path.exists(data_path):
        print(f"⚠️ [跳过] 未检测到精度阶梯文件: {data_path}")
        continue
        
    print(f"\n⚡ [当前进度] 正在全面计算精度级别 -> Acc = {acc_str}")
    data = np.load(data_path)
    Y_total = data["Y"]
    Yhat_total = data["Y_hat"] if "Y_hat" in data.files else data["Yhat"]
    
    grid = np.concatenate([Y_total, Yhat_total], axis=0)
    grid = np.linspace(grid.min(), grid.max(), 5000)
    
    # --------------------------------------------------------------------------
    # 3.1 运行当前精度下的标准 IID 实验并画图
    # --------------------------------------------------------------------------
    dataset_dist_iid = 'IID'
    num_ratio_iid = [1] * num_clients
    
    true_theta_iid, cpp_intervals_iid, ppi_ci_combined_iid, mean_cpp_iid = analyze_dataset(
        alpha, None, Y_total, Yhat_total, dataset_dist_iid, num_ratio_iid, method, grid
    )
    title_iid = f"Acc = {acc_str}"
    plot_cpp(true_theta_iid, cpp_intervals_iid, ppi_ci_combined_iid, mean_cpp_iid, 
             dataset_name, dataset_dist_iid, acc_str, None, xlim, ylim, title_iid, None)
    
    iid_total_records.append({
        'dataset': dataset_name, 'target_accuracy': acc_str, 'distribution': dataset_dist_iid, 'alpha_dir': 'None',
        'ppi_ci_lower': ppi_ci_combined_iid[0], 'ppi_ci_upper': ppi_ci_combined_iid[1], 'mean_cpp_lower': mean_cpp_iid[0], 'mean_cpp_upper': mean_cpp_iid[1]
    })
    
    # --------------------------------------------------------------------------
    # 3.2 运行当前精度下的三种 alpha_Dir Non-IID 实验并画图
    # --------------------------------------------------------------------------
    alpha_dir_list = [1.0, 0.1, 0.01]
    dataset_dist_non = 'Non-IID'
    for alpha_dir in alpha_dir_list:
        Y_dir, Yhat_dir, actual_sizes = dirichlet_continuous_allocation(
            Y_total, Yhat_total, alpha_dir, num_clients, min_samples_per_client=15
        )
        true_theta_dir, cpp_intervals_dir, ppi_ci_combined_dir, mean_cpp_dir = analyze_dataset(
            alpha, None, Y_dir, Yhat_dir, 'pre_split', actual_sizes, method, grid
        )
        # 🛠️ 核心更正：恢复被漏掉的动态 Acc 精度后缀
        title_dir = f"$\\alpha_{{Dir}} = {alpha_dir}$ (Acc = {acc_str})"
        plot_cpp(true_theta_dir, cpp_intervals_dir, ppi_ci_combined_dir, mean_cpp_dir, 
                 dataset_name, dataset_dist_non, acc_str, alpha_dir, xlim, ylim, title_dir, None)
        
        non_iid_total_records.append({
            'dataset': dataset_name, 'target_accuracy': acc_str, 'distribution': dataset_dist_non, 'alpha_dir': str(alpha_dir),
            'ppi_ci_lower': ppi_ci_combined_dir[0], 'ppi_ci_upper': ppi_ci_combined_dir[1], 'mean_cpp_lower': mean_cpp_dir[0], 'mean_cpp_upper': mean_cpp_dir[1]
        })

# CSV 指标落盘
csv_flat_dir = os.path.join('.', 'result', dataset_name, 'csv')
os.makedirs(csv_flat_dir, exist_ok=True)
if len(iid_total_records) > 0:
    pd.DataFrame(iid_total_records).to_csv(os.path.join(csv_flat_dir, f'{dataset_name}_IID_summary.csv'), index=False)
if len(non_iid_total_records) > 0:
    pd.DataFrame(non_iid_total_records).to_csv(os.path.join(csv_flat_dir, f'{dataset_name}_Non-IID_summary.csv'), index=False)


⚡ [当前进度] 正在全面计算精度级别 -> Acc = 0.1
labeled_ratio 0.3
分组： 1
带标签的样本量： 11
不带标签的样本量： 27
分组： 2
带标签的样本量： 11
不带标签的样本量： 27
分组： 3
带标签的样本量： 11
不带标签的样本量： 26
分组： 4
带标签的样本量： 11
不带标签的样本量： 26
分组： 5
带标签的样本量： 11
不带标签的样本量： 26
分组： 6
带标签的样本量： 11
不带标签的样本量： 26
分组： 7
带标签的样本量： 11
不带标签的样本量： 26
分组： 8
带标签的样本量： 11
不带标签的样本量： 26
分组： 9
带标签的样本量： 11
不带标签的样本量： 26
分组： 10
带标签的样本量： 11
不带标签的样本量： 26
分组： 11
带标签的样本量： 11
不带标签的样本量： 26
分组： 12
带标签的样本量： 11
不带标签的样本量： 26
分组： 13
带标签的样本量： 11
不带标签的样本量： 26
分组： 14
带标签的样本量： 11
不带标签的样本量： 26
分组： 15
带标签的样本量： 11
不带标签的样本量： 26
分组： 16
带标签的样本量： 11
不带标签的样本量： 26
分组： 17
带标签的样本量： 11
不带标签的样本量： 26
分组： 18
带标签的样本量： 11
不带标签的样本量： 26
分组： 19
带标签的样本量： 11
不带标签的样本量： 26
分组： 20
带标签的样本量： 11
不带标签的样本量： 26
带标签的样本量： 220
不带标签的样本量： 522

最终结果：
真实 theta: 97.5
CPP intervals: [array([ 76.02611479, 101.71641981]), array([ 62.50734669, 104.13214469]), array([ 76.35130852, 120.48474391]), array([ 70.54427755, 111.47223184]), array([ 86.01420806, 121.97134383]), array([ 75.51509606, 107.47699453]), array([ 85.50318933, 112.49426

In [9]:
# %%
# ==============================================================================
# 1. 固定准确率为 50% 的基准数据集调入与 grid 初始化
# ==============================================================================
fixed_acc_str = '0.5'
data_path = f'../data/{dataset_name}/{dataset_name}_acc_{fixed_acc_str}.npz'

print(f"正在读取固定 50% 准确率的目标基准数据: {data_path}")
data = np.load(data_path)
Y_total = data["Y"]
Yhat_total = data["Y_hat"] if "Y_hat" in data.files else data["Yhat"]

grid = np.concatenate([Y_total, Yhat_total], axis=0)
grid = np.linspace(grid.min(), grid.max(), 5000)

iid_ratio_records = []
non_iid_ratio_records = []
ratio_steps = [0.1, 0.2, 0.3, 0.4, 0.5]

# ==============================================================================
# 2. 纵向标签比例变动大循环核心
# ==============================================================================
for ratio in ratio_steps:
    ratio_str = f"{ratio:.1f}"
    sub_folder_name = f"ratio_{ratio_str}"  
    print(f"\n🚀 [实验进行中] 正在注入比例阶梯 -> labeled_ratio = {ratio_str}")
    
    # 4.1 变比率 IID 支线
    dataset_dist_iid = 'IID'
    num_ratio_iid = [1] * num_clients
    true_theta_iid, cpp_intervals_iid, ppi_ci_combined_iid, mean_cpp_iid = analyze_dataset(
        alpha, None, Y_total, Yhat_total, dataset_dist_iid, num_ratio_iid, method, grid, current_ratio=ratio
    )
    # 🛠️ 核心更正：IID 轴标对齐
    title_iid = f"$\\lambda = {ratio_str}$ (Acc = 0.5)"
    plot_cpp(true_theta_iid, cpp_intervals_iid, ppi_ci_combined_iid, mean_cpp_iid, 
             dataset_name, dataset_dist_iid, fixed_acc_str, None, xlim, ylim, title_iid, sub_folder_name)
    
    iid_ratio_records.append({
        'dataset': dataset_name, 'fixed_accuracy': fixed_acc_str, 'labeled_ratio': ratio_str, 'distribution': dataset_dist_iid, 'alpha_dir': 'None',
        'ppi_ci_lower': ppi_ci_combined_iid[0], 'ppi_ci_upper': ppi_ci_combined_iid[1], 'mean_cpp_lower': mean_cpp_iid[0], 'mean_cpp_upper': mean_cpp_iid[1]
    })
    
    # 4.2 变比率 Non-IID 支线
    dataset_dist_non = 'Non-IID'
    alpha_dir_list = [1.0, 0.1, 0.01]
    for alpha_dir in alpha_dir_list:
        Y_dir, Yhat_dir, actual_sizes = dirichlet_continuous_allocation(
            Y_total, Yhat_total, alpha_dir, num_clients, min_samples_per_client=15
        )
        true_theta_dir, cpp_intervals_dir, ppi_ci_combined_dir, mean_cpp_dir = analyze_dataset(
            alpha, None, Y_dir, Yhat_dir, 'pre_split', actual_sizes, method, grid, current_ratio=ratio
        )
        # 🛠️ 核心更正：恢复少样本抽样比 \lambda 后缀
        title_dir = f"$\\alpha_{{Dir}} = {alpha_dir}$ ($\\lambda = {ratio_str}$)"
        plot_cpp(true_theta_dir, cpp_intervals_dir, ppi_ci_combined_dir, mean_cpp_dir, 
                 dataset_name, dataset_dist_non, fixed_acc_str, alpha_dir, xlim, ylim, title_dir, sub_folder_name)
        
        non_iid_ratio_records.append({
            'dataset': dataset_name, 'fixed_accuracy': fixed_acc_str, 'labeled_ratio': ratio_str, 'distribution': dataset_dist_non, 'alpha_dir': str(alpha_dir),
            'ppi_ci_lower': ppi_ci_combined_dir[0], 'ppi_ci_upper': ppi_ci_combined_dir[1], 'mean_cpp_lower': mean_cpp_dir[0], 'mean_cpp_upper': mean_cpp_dir[1]
        })

# 跨比率 master csv 焊接入库
if len(iid_ratio_records) > 0:
    pd.DataFrame(iid_ratio_records).to_csv(os.path.join(csv_flat_dir, f'{dataset_name}_IID_ratio_summary.csv'), index=False)
if len(non_iid_ratio_records) > 0:
    pd.DataFrame(non_iid_ratio_records).to_csv(os.path.join(csv_flat_dir, f'{dataset_name}_Non-IID_ratio_summary.csv'), index=False)

正在读取固定 50% 准确率的目标基准数据: ../data/salary/salary_acc_0.5.npz

🚀 [实验进行中] 正在注入比例阶梯 -> labeled_ratio = 0.1
labeled_ratio 0.1
分组： 1
带标签的样本量： 3
不带标签的样本量： 35
分组： 2
带标签的样本量： 3
不带标签的样本量： 35
分组： 3
带标签的样本量： 3
不带标签的样本量： 34
分组： 4
带标签的样本量： 3
不带标签的样本量： 34
分组： 5
带标签的样本量： 3
不带标签的样本量： 34
分组： 6
带标签的样本量： 3
不带标签的样本量： 34
分组： 7
带标签的样本量： 3
不带标签的样本量： 34
分组： 8
带标签的样本量： 3
不带标签的样本量： 34
分组： 9
带标签的样本量： 3
不带标签的样本量： 34
分组： 10
带标签的样本量： 3
不带标签的样本量： 34
分组： 11
带标签的样本量： 3
不带标签的样本量： 34
分组： 12
带标签的样本量： 3
不带标签的样本量： 34
分组： 13
带标签的样本量： 3
不带标签的样本量： 34
分组： 14
带标签的样本量： 3
不带标签的样本量： 34
分组： 15
带标签的样本量： 3
不带标签的样本量： 34
分组： 16
带标签的样本量： 3
不带标签的样本量： 34
分组： 17
带标签的样本量： 3
不带标签的样本量： 34
分组： 18
带标签的样本量： 3
不带标签的样本量： 34
分组： 19
带标签的样本量： 3
不带标签的样本量： 34
分组： 20
带标签的样本量： 3
不带标签的样本量： 34
带标签的样本量： 60
不带标签的样本量： 682

最终结果：
真实 theta: 97.5
CPP intervals: [array([ 31.54110822, 100.09731946]), array([ 27.5480096 , 100.09731946]), array([ 96.0080016 , 100.96329266]), array([ 96.0080016 , 129.49229846]), array([ 87.54070814, 100.09731946]), array([ 87.54070814, 1